Savitzky-Golay filter

In [1]:
import os
import pandas as pd
from scipy.signal import savgol_filter

# Input and output directories
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL'
output_directory = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Output_Data/Filtered_Data_NSW_ALL'

#window_length=5, polyorder=2
#window_length=9, polyorder=2
#polyorder=3

# Ensure the output directory exists
os.makedirs(output_directory, exist_ok=True)

# Function to apply Savitzky-Golay filter to the data
def apply_savgol_filter(file_path, output_path, window_length=5, polyorder=2):
    # Read the CSV file
    df = pd.read_csv(file_path)

    # Apply Savitzky-Golay filter to each column except Date, Time, and PM2.5
    for column in df.columns:
        if column not in ['Date', 'Time', 'PM2.5']:
            df[column] = savgol_filter(df[column], window_length=window_length, polyorder=polyorder)

    # Save the modified DataFrame to the output directory
    df.to_csv(output_path, index=False)

# Iterate over all CSV files in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        input_file = os.path.join(input_directory, filename)
        output_file = os.path.join(output_directory, filename)
        
        # Apply Savitzky-Golay filter and save the results
        apply_savgol_filter(input_file, output_file)

print(f"All files processed and saved in {output_directory}")


All files processed and saved in /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Output_Data/Filtered_Data_NSW_ALL


Wavelet Transform Denoising:

Fourier Transform Denoising

In [8]:
import os
import pandas as pd
import numpy as np

# Input and output directories
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL'
output_directory = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Output_Data/Filtered_Data_NSW_ALL_Fourier'

# Ensure the output directory exists
os.makedirs(output_directory, exist_ok=True)

# Function to apply Fourier Transform Denoising to the data
def apply_fourier_denoising(file_path, output_path, cutoff_frequency=0.1):
    # Read the CSV file
    df = pd.read_csv(file_path)

    # Define a function to apply Fourier denoising to a single column
    def fourier_denoise(data, cutoff_frequency):
        # Ensure the data is numeric and clean NaN values
        data = pd.to_numeric(data, errors='coerce').fillna(0)

        # Perform Fast Fourier Transform (FFT)
        fft_data = np.fft.fft(data)

        # Get frequencies corresponding to FFT components
        fft_freq = np.fft.fftfreq(len(data))

        # Apply low-pass filter: set high-frequency components to zero
        fft_data[np.abs(fft_freq) > cutoff_frequency] = 0

        # Perform inverse FFT to get the denoised signal
        denoised_data = np.fft.ifft(fft_data)

        # Return real part of the inverse FFT (denoised signal)
        return denoised_data.real

    # Apply Fourier Denoising to each column except Date, Time, and PM2.5
    for column in df.columns:
        if column not in ['Date', 'Time', 'PM2.5']:
            try:
                df[column] = fourier_denoise(df[column], cutoff_frequency)
            except ValueError as e:
                print(f"Error applying Fourier denoising to column {column} in file {file_path}: {e}")

    # Save the modified DataFrame to the output directory
    df.to_csv(output_path, index=False)

# Iterate over all CSV files in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        input_file = os.path.join(input_directory, filename)
        output_file = os.path.join(output_directory, filename)

        # Apply Fourier Transform Denoising and save the results
        apply_fourier_denoising(input_file, output_file)

print(f"All files processed and saved in {output_directory}")


All files processed and saved in /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Output_Data/Filtered_Data_NSW_ALL_Fourier


Autoencoders for Denoising

In [ ]:
import os
import pandas as pd
import numpy as np
from keras.models import Model
from keras.layers import Input, Dense
from sklearn.preprocessing import MinMaxScaler

# Input and output directories
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL'
output_directory = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Output_Data/Filtered_Data_NSW_ALL_Autoencoder'

# Ensure the output directory exists
os.makedirs(output_directory, exist_ok=True)

# Function to build the autoencoder model
def build_autoencoder(input_dim):
    # Input layer
    input_layer = Input(shape=(input_dim,))
    
    # Encoder layers
    encoded = Dense(128, activation='relu')(input_layer)
    encoded = Dense(64, activation='relu')(encoded)
    
    # Latent space
    encoded = Dense(32, activation='relu')(encoded)
    
    # Decoder layers
    decoded = Dense(64, activation='relu')(encoded)
    decoded = Dense(128, activation='relu')(decoded)
    
    # Output layer (reconstructed input)
    decoded = Dense(input_dim, activation='sigmoid')(decoded)
    
    # Autoencoder model
    autoencoder = Model(input_layer, decoded)
    
    # Compile the model
    autoencoder.compile(optimizer='adam', loss='mse')
    
    return autoencoder

# Function to apply Autoencoder Denoising to the data
def apply_autoencoder_denoising(file_path, output_path):
    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Extract the columns to denoise (exclude Date, Time, and PM2.5)
    columns_to_denoise = [col for col in df.columns if col not in ['Date', 'Time', 'PM2.5']]
    data = df[columns_to_denoise]
    
    # Normalize the data using MinMaxScaler
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)
    
    # Build and train the autoencoder
    input_dim = data_scaled.shape[1]
    autoencoder = build_autoencoder(input_dim)
    
    # Train the autoencoder on the scaled data
    autoencoder.fit(data_scaled, data_scaled, epochs=50, batch_size=32, shuffle=True, verbose=0)
    
    # Use the trained autoencoder to denoise the data
    denoised_data = autoencoder.predict(data_scaled)
    
    # Inverse transform the scaled data to original scale
    denoised_data = scaler.inverse_transform(denoised_data)
    
    # Place the denoised data back into the DataFrame
    df[columns_to_denoise] = denoised_data
    
    # Save the modified DataFrame to the output directory
    df.to_csv(output_path, index=False)

# Iterate over all CSV files in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        input_file = os.path.join(input_directory, filename)
        output_file = os.path.join(output_directory, filename)
        
        # Apply Autoencoder Denoising and save the results
        apply_autoencoder_denoising(input_file, output_file)

print(f"All files processed and saved in {output_directory}")
